# RAF-DB full test: five model paths (Google Colab)

This notebook evaluates POSTER, MA3DNRe, MA3DNCo, raw MA3D, and MA3DReCo on all 3,068 RAF-DB test images. It uses the number of Colab GPUs available (up to two), keeps only concise cell output, and writes resumable numeric result files to Google Drive.

## Before running

1. Upload the complete `ggcolab/` folder to `MyDrive/ggcolab/`. Keep `bundle/` and `scripts/` together. The private bundle contains RAF-DB images and model weights; do not make it public.
2. Open this notebook in Colab. In **Runtime → Change runtime type**, select a GPU, then run the cells from top to bottom. Internet access is needed for package installation and the pinned EMOCA source/assets.
3. The first setup downloads and builds the legacy EMOCA environment. Register and personally accept FLAME/EMOCA terms before the asset-download cell runs. The consent flag defaults to `True` as requested.
4. The smoke pass runs two images before the full test. Results are saved under `MyDrive/ggcolab/results/`; rerunning can resume completed rows.

The Colab runtime and its local disk are temporary. A runtime reset requires repeating setup and the gated asset download; completed result rows remain in Drive. Cell output is quiet on success; the final summary shows metrics, and a failure shows only the tail of its log.

The worker keeps Colab's preinstalled CUDA PyTorch and NumPy. EMOCA runs in a separate Python 3.8 / Torch 1.11 environment. The benchmark writes predictions and metrics only, not input, cropped, or restored images.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

def run_quiet(command, *, label, env=None):
    completed = subprocess.run(
        command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors='replace',
    )
    if completed.returncode:
        tail = (completed.stdout or '').strip()[-4000:]
        raise RuntimeError(f'{label} failed (exit {completed.returncode}): {tail}')

# Change this if you placed the uploaded folder elsewhere in Drive.
DRIVE_ROOT = Path('/content/drive/MyDrive/ggcolab')
# The project owner has already registered and accepted FLAME/EMOCA terms for this research run.
ACCEPT_FLAME_EMOCA_LICENSES = True
RUN_FULL_BENCHMARK = True
MAX_GPUS = 2

ROOT = Path('/content/rafdb5')
WORKDIR = ROOT / 'runtime'
LOCAL_BUNDLE = ROOT / 'bundle'
SCRIPTS = ROOT / 'scripts'
DRIVE_RESULTS = DRIVE_ROOT / 'results'
WORKDIR.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)

import torch
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'
GPU_COUNT = min(torch.cuda.device_count(), MAX_GPUS)
print(f"GPU workers: {GPU_COUNT} ({torch.cuda.get_device_name(0)})")
assert (DRIVE_ROOT / 'bundle' / 'BUNDLE_MANIFEST.json').is_file(), (
    f"Could not find {DRIVE_ROOT / 'bundle'}. Upload the complete ggcolab folder to Drive."
)


In [ ]:
# Copy the ~1.2 GB bundle from Drive to Colab's local disk for faster image/model access.
shutil.copytree(DRIVE_ROOT / 'bundle', LOCAL_BUNDLE, dirs_exist_ok=True)
shutil.copytree(DRIVE_ROOT / 'scripts', SCRIPTS, dirs_exist_ok=True)
assert (SCRIPTS / 'prepare_runtime.py').is_file(), 'Missing benchmark scripts beside ggcolab/scripts.'

# Install only the worker packages; do not replace Colab's CUDA-enabled torch or NumPy.
requirements = LOCAL_BUNDLE / 'environment' / 'requirements-worker.txt'
runtime_requirements = WORKDIR / 'requirements-worker-colab.txt'
requirements_lines = requirements.read_text(encoding='utf-8').splitlines()
runtime_requirements.write_text(
    '\n'.join(line for line in requirements_lines
                if not line.strip().lower().startswith('numpy==')) + '\n',
    encoding='utf-8',
)
run_quiet([sys.executable, '-m', 'pip', 'install', '-r', str(runtime_requirements)],
          label='Worker package installation')
run_quiet([sys.executable, str(SCRIPTS / 'verify_bundle.py'), str(LOCAL_BUNDLE)],
          label='Bundle verification')

run_quiet([sys.executable, str(SCRIPTS / 'prepare_runtime.py'),
           '--bundle-root', str(LOCAL_BUNDLE), '--workdir', str(WORKDIR)],
          label='Runtime staging')
# Older Drive bundles may predate M6 and omit this module, while the
# packaged image-preprocessing __init__ imports it eagerly.
import base64
m6_pipeline = WORKDIR / 'project' / 'services' / 'image_preprocessing' / 'pipeline.py'
m6_pipeline.parent.mkdir(parents=True, exist_ok=True)
m6_pipeline.write_bytes(base64.b64decode('ZnJvbSBfX2Z1dHVyZV9fIGltcG9ydCBhbm5vdGF0aW9ucwpmcm9tIGRhdGFjbGFzc2VzIGltcG9ydCBkYXRhY2xhc3MsIGZpZWxkCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHRpbWUgaW1wb3J0IHBlcmZfY291bnRlcgpmcm9tIHR5cGluZyBpbXBvcnQgQW55LCBQcm90b2NvbAppbXBvcnQgbWF0aAppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBQSUwgaW1wb3J0IEltYWdlLCBJbWFnZU9wcwoKY2xhc3MgRGV0ZWN0b3IoUHJvdG9jb2wpOgogICAgbW9kZWxfaWQ6IHN0cgogICAgZGVmIGRldGVjdChzZWxmLCBpbWFnZTogSW1hZ2UuSW1hZ2UsIGNvbmZpZGVuY2U6IGZsb2F0KSAtPiBsaXN0W2RpY3Rbc3RyLCBBbnldXTogLi4uCmNsYXNzIFJlc3RvcmVyKFByb3RvY29sKToKICAgIG1vZGVsX2lkOiBzdHIKICAgIGRlZiByZXN0b3JlKHNlbGYsIGltYWdlOiBJbWFnZS5JbWFnZSwgZmlkZWxpdHk6IGZsb2F0IHwgTm9uZSA9IE5vbmUpIC0+IEltYWdlLkltYWdlOiAuLi4KCkBkYXRhY2xhc3MKY2xhc3MgUGlwZWxpbmVDb25maWc6CiAgICBjb25maWRlbmNlOiBmbG9hdCA9IDAuNQogICAgcGFkZGluZzogZmxvYXQgPSAwLjE1CiAgICBmaWRlbGl0eTogZmxvYXQgPSAwLjcKICAgIG1heF9zaWRlOiBpbnQgPSA0MDk2CiAgICBub19mYWNlX3BvbGljeTogc3RyID0gImtlZXAiCiAgICBudW1weV9jb2xvcjogc3RyID0gIlJHQiIKQGRhdGFjbGFzcwpjbGFzcyBGYWNlUmVzdWx0OgogICAgYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNyb3BfYmJveF94eXh5OiBsaXN0W2ludF0KICAgIGNvbmZpZGVuY2U6IGZsb2F0CiAgICBjcm9wOiBJbWFnZS5JbWFnZQpAZGF0YWNsYXNzCmNsYXNzIFBpcGVsaW5lUmVzdWx0OgogICAgZW5oYW5jZWQ6IEltYWdlLkltYWdlIHwgTm9uZQogICAgZmFjZXM6IGxpc3RbRmFjZVJlc3VsdF0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9bGlzdCkKICAgIGlucHV0X2RpbWVuc2lvbnM6IGxpc3RbaW50XSB8IE5vbmUgPSBOb25lCiAgICBvdXRwdXRfZGltZW5zaW9uczogbGlzdFtpbnRdIHwgTm9uZSA9IE5vbmUKICAgIG1vZGVsX2lkczogZGljdFtzdHIsIHN0cl0gPSBmaWVsZChkZWZhdWx0X2ZhY3Rvcnk9ZGljdCkKICAgIHN0YWdlX3RpbWVzX21zOiBkaWN0W3N0ciwgZmxvYXRdID0gZmllbGQoZGVmYXVsdF9mYWN0b3J5PWRpY3QpCiAgICBzdGF0dXM6IHN0ciA9ICJlcnJvciIKICAgIGVycm9yOiBzdHIgfCBOb25lID0gTm9uZQoKZGVmIF9pbnB1dF9yZ2IoaW1hZ2U6IHN0ciB8IFBhdGggfCBJbWFnZS5JbWFnZSB8IG5wLm5kYXJyYXksIG51bXB5X2NvbG9yOiBzdHIpIC0+IEltYWdlLkltYWdlOgogICAgaWYgaXNpbnN0YW5jZShpbWFnZSwgKHN0ciwgUGF0aCkpOgogICAgICAgIHdpdGggSW1hZ2Uub3BlbihpbWFnZSkgYXMgc3JjOiByZXR1cm4gSW1hZ2VPcHMuZXhpZl90cmFuc3Bvc2Uoc3JjKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIEltYWdlLkltYWdlKTogcmV0dXJuIEltYWdlT3BzLmV4aWZfdHJhbnNwb3NlKGltYWdlKS5jb252ZXJ0KCJSR0IiKS5jb3B5KCkKICAgIGlmIGlzaW5zdGFuY2UoaW1hZ2UsIG5wLm5kYXJyYXkpOgogICAgICAgIGlmIGltYWdlLmR0eXBlICE9IG5wLnVpbnQ4IG9yIGltYWdlLm5kaW0gIT0gMyBvciBpbWFnZS5zaGFwZVsyXSAhPSAzIG9yIG1pbihpbWFnZS5zaGFwZVs6Ml0pIDwgMToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigiTnVtUHkgaW5wdXQgbXVzdCBiZSBub24tZW1wdHkgSHhXeDMgdWludDgiKQogICAgICAgIGlmIG51bXB5X2NvbG9yIG5vdCBpbiAoIlJHQiIsICJCR1IiKTogcmFpc2UgVmFsdWVFcnJvcigibnVtcHlfY29sb3IgbXVzdCBiZSBSR0Igb3IgQkdSIikKICAgICAgICBhcnIgPSBpbWFnZSBpZiBudW1weV9jb2xvciA9PSAiUkdCIiBlbHNlIGltYWdlWy4uLiwgOjotMV0KICAgICAgICByZXR1cm4gSW1hZ2UuZnJvbWFycmF5KG5wLmFzY29udGlndW91c2FycmF5KGFyciksICJSR0IiKQogICAgcmFpc2UgVHlwZUVycm9yKCJpbnB1dCBtdXN0IGJlIGEgcGF0aCwgUElMIGltYWdlLCBvciBOdW1QeSBSR0IgYXJyYXkiKQoKZGVmIF94eXh5KGJveDogQW55LCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCkgLT4gbGlzdFtpbnRdOgogICAgdiA9IG5wLmFzYXJyYXkoYm94LCBkdHlwZT1ucC5mbG9hdDY0KS5yZXNoYXBlKC0xKQogICAgaWYgbGVuKHYpICE9IDQgb3Igbm90IG5wLmlzZmluaXRlKHYpLmFsbCgpOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciByZXR1cm5lZCBtYWxmb3JtZWQgYmJveF94eXh5IikKICAgIHgxLCB5MSwgeDIsIHkyID0gdgogICAgeDEsIHgyID0gc29ydGVkKChpbnQobnAuZmxvb3IoeDEpKSwgaW50KG5wLmNlaWwoeDIpKSkpCiAgICB5MSwgeTIgPSBzb3J0ZWQoKGludChucC5mbG9vcih5MSkpLCBpbnQobnAuY2VpbCh5MikpKSkKICAgIHgxLCB4MiA9IG1heCgwLCBtaW4od2lkdGgsIHgxKSksIG1heCgwLCBtaW4od2lkdGgsIHgyKSkKICAgIHkxLCB5MiA9IG1heCgwLCBtaW4oaGVpZ2h0LCB5MSkpLCBtYXgoMCwgbWluKGhlaWdodCwgeTIpKQogICAgaWYgeDIgPD0geDEgb3IgeTIgPD0geTE6IHJhaXNlIFZhbHVlRXJyb3IoImRldGVjdG9yIHJldHVybmVkIGVtcHR5IGJib3giKQogICAgcmV0dXJuIFt4MSwgeTEsIHgyLCB5Ml0KCmRlZiBfcGFkZGVkKGJveDogbGlzdFtpbnRdLCB3aWR0aDogaW50LCBoZWlnaHQ6IGludCwgcGFkZGluZzogZmxvYXQpIC0+IGxpc3RbaW50XToKICAgIGlmIG5vdCBtYXRoLmlzZmluaXRlKHBhZGRpbmcpIG9yIHBhZGRpbmcgPCAwOiByYWlzZSBWYWx1ZUVycm9yKCJwYWRkaW5nIG11c3QgYmUgZmluaXRlIGFuZCBub24tbmVnYXRpdmUiKQogICAgeDEseTEseDIseTIgPSBib3g7IGJ3LGJoPXgyLXgxLHkyLXkxOyBjeCxjeT0oeDEreDIpLzIsKHkxK3kyKS8yCiAgICByZXR1cm4gW21heCgwLGludChucC5mbG9vcihjeC1idyooLjUrcGFkZGluZykpKSksIG1heCgwLGludChucC5mbG9vcihjeS1iaCooLjUrcGFkZGluZykpKSksCiAgICAgICAgICAgIG1pbih3aWR0aCxpbnQobnAuY2VpbChjeCtidyooLjUrcGFkZGluZykpKSksIG1pbihoZWlnaHQsaW50KG5wLmNlaWwoY3krYmgqKC41K3BhZGRpbmcpKSkpXQoKZGVmIF9saW1pdChpbWFnZTogSW1hZ2UuSW1hZ2UsIG1heF9zaWRlOiBpbnQpIC0+IEltYWdlLkltYWdlOgogICAgaWYgbWF4X3NpZGUgPCAxOiByYWlzZSBWYWx1ZUVycm9yKCJtYXhfc2lkZSBtdXN0IGJlIHBvc2l0aXZlIikKICAgIHcsaD1pbWFnZS5zaXplOyBzY2FsZT1taW4oMS4sbWF4X3NpZGUvbWF4KHcsaCkpCiAgICByZXR1cm4gaW1hZ2UgaWYgc2NhbGUgPT0gMSBlbHNlIGltYWdlLnJlc2l6ZSgobWF4KDEsaW50KHcqc2NhbGUpKSxtYXgoMSxpbnQoaCpzY2FsZSkpKSxJbWFnZS5SZXNhbXBsaW5nLkxBTkNaT1MpCgpjbGFzcyBJbWFnZVBpcGVsaW5lOiAgIyBzdGFuZGFsb25lIE02IHByb2Nlc3Npbmcgb3JjaGVzdHJhdGlvbgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGdlbmVyYWxfcmVzdG9yZXI6IFJlc3RvcmVyLCBkZXRlY3RvcjogRGV0ZWN0b3IsIGZhY2VfcmVzdG9yZXI6IFJlc3RvcmVyLCBjb25maWc6IFBpcGVsaW5lQ29uZmlnfE5vbmU9Tm9uZSk6CiAgICAgICAgc2VsZi5nZW5lcmFsX3Jlc3RvcmVyLHNlbGYuZGV0ZWN0b3Isc2VsZi5mYWNlX3Jlc3RvcmVyPWdlbmVyYWxfcmVzdG9yZXIsZGV0ZWN0b3IsZmFjZV9yZXN0b3JlcgogICAgICAgIHNlbGYuY29uZmlnPWNvbmZpZyBvciBQaXBlbGluZUNvbmZpZygpCiAgICAgICAgaWYgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgbm90IGluICgia2VlcCIsImVycm9yIik6IHJhaXNlIFZhbHVlRXJyb3IoIm5vX2ZhY2VfcG9saWN5IG11c3QgYmUga2VlcCBvciBlcnJvciIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcucGFkZGluZykgb3Igc2VsZi5jb25maWcucGFkZGluZyA8IDA6IHJhaXNlIFZhbHVlRXJyb3IoInBhZGRpbmcgbXVzdCBiZSBmaW5pdGUgYW5kIG5vbi1uZWdhdGl2ZSIpCiAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoc2VsZi5jb25maWcuY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJjb25maWRlbmNlIG11c3QgYmUgaW4gWzAsIDFdIikKICAgICAgICBpZiBub3QgbWF0aC5pc2Zpbml0ZShzZWxmLmNvbmZpZy5maWRlbGl0eSkgb3Igbm90IDAgPD0gc2VsZi5jb25maWcuZmlkZWxpdHkgPD0gMTogcmFpc2UgVmFsdWVFcnJvcigiZmlkZWxpdHkgbXVzdCBiZSBpbiBbMCwgMV0iKQogICAgICAgIGlmIHNlbGYuY29uZmlnLm1heF9zaWRlIDwgMTogcmFpc2UgVmFsdWVFcnJvcigibWF4X3NpZGUgbXVzdCBiZSBwb3NpdGl2ZSIpCiAgICAgICAgaWYgc2VsZi5jb25maWcubnVtcHlfY29sb3Igbm90IGluICgiUkdCIiwgIkJHUiIpOiByYWlzZSBWYWx1ZUVycm9yKCJudW1weV9jb2xvciBtdXN0IGJlIFJHQiBvciBCR1IiKQogICAgZGVmIHByb2Nlc3Moc2VsZiwgc291cmNlOiBzdHJ8UGF0aHxJbWFnZS5JbWFnZXxucC5uZGFycmF5KSAtPiBQaXBlbGluZVJlc3VsdDoKICAgICAgICByPVBpcGVsaW5lUmVzdWx0KGVuaGFuY2VkPU5vbmUsbW9kZWxfaWRzPXsiZ2VuZXJhbCI6c2VsZi5nZW5lcmFsX3Jlc3RvcmVyLm1vZGVsX2lkLCJkZXRlY3RvciI6c2VsZi5kZXRlY3Rvci5tb2RlbF9pZCwiZmFjZSI6c2VsZi5mYWNlX3Jlc3RvcmVyLm1vZGVsX2lkfSkKICAgICAgICB0cnk6CiAgICAgICAgICAgIG9yaWdpbmFsPV9pbnB1dF9yZ2Ioc291cmNlLHNlbGYuY29uZmlnLm51bXB5X2NvbG9yKTsgci5pbnB1dF9kaW1lbnNpb25zPVtvcmlnaW5hbC53aWR0aCxvcmlnaW5hbC5oZWlnaHRdCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IGVuaGFuY2VkPXNlbGYuZ2VuZXJhbF9yZXN0b3Jlci5yZXN0b3JlKG9yaWdpbmFsLmNvcHkoKSk7IHIuc3RhZ2VfdGltZXNfbXNbImdlbmVyYWxfcmVzdG9yYXRpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShlbmhhbmNlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZ2VuZXJhbCByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICBlbmhhbmNlZD1lbmhhbmNlZC5jb252ZXJ0KCJSR0IiKQogICAgICAgICAgICBpZiBlbmhhbmNlZC5zaXplICE9IG9yaWdpbmFsLnNpemU6IHJhaXNlIFZhbHVlRXJyb3IoImdlbmVyYWwgcmVzdG9yYXRpb24gbXVzdCBwcmVzZXJ2ZSBkaW1lbnNpb25zOyBiYm94IG1hcHBpbmcgaXMgdW5kZWZpbmVkIikKICAgICAgICAgICAgci5lbmhhbmNlZD1lbmhhbmNlZDsgci5vdXRwdXRfZGltZW5zaW9ucz1bZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0XQogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpOyBkZXRlY3Rpb25zPXNlbGYuZGV0ZWN0b3IuZGV0ZWN0KGVuaGFuY2VkLmNvcHkoKSxzZWxmLmNvbmZpZy5jb25maWRlbmNlKTsgci5zdGFnZV90aW1lc19tc1siZmFjZV9kZXRlY3Rpb24iXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBkIGluIGRldGVjdGlvbnM6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgY29uZmlkZW5jZT1mbG9hdChkWyJjb25maWRlbmNlIl0pCiAgICAgICAgICAgICAgICAgICAgaWYgbm90IG1hdGguaXNmaW5pdGUoY29uZmlkZW5jZSkgb3Igbm90IDAgPD0gY29uZmlkZW5jZSA8PSAxOiByYWlzZSBWYWx1ZUVycm9yKCJkZXRlY3RvciBjb25maWRlbmNlIG11c3QgYmUgZmluaXRlIGFuZCBpbiBbMCwgMV0iKQogICAgICAgICAgICAgICAgICAgIGlmIGNvbmZpZGVuY2UgPCBzZWxmLmNvbmZpZy5jb25maWRlbmNlOiBjb250aW51ZQogICAgICAgICAgICAgICAgICAgIGJveD1feHl4eShkLmdldCgiYmJveF94eXh5IiksZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0KTsgY2I9X3BhZGRlZChib3gsZW5oYW5jZWQud2lkdGgsZW5oYW5jZWQuaGVpZ2h0LHNlbGYuY29uZmlnLnBhZGRpbmcpCiAgICAgICAgICAgICAgICAgICAgci5mYWNlcy5hcHBlbmQoRmFjZVJlc3VsdChib3gsY2IsY29uZmlkZW5jZSxlbmhhbmNlZC5jcm9wKHR1cGxlKGNiKSkuY29udmVydCgiUkdCIikpKQogICAgICAgICAgICAgICAgZXhjZXB0IChLZXlFcnJvciwgVHlwZUVycm9yLCBWYWx1ZUVycm9yKSBhcyBleGM6CiAgICAgICAgICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImRldGVjdG9yIHJldHVybmVkIG1hbGZvcm1lZCBmYWNlOiB7ZXhjfSIpIGZyb20gZXhjCiAgICAgICAgICAgIHIuc3RhZ2VfdGltZXNfbXNbImZhY2VfY3JvcHBpbmciXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICB0PXBlcmZfY291bnRlcigpCiAgICAgICAgICAgIGZvciBmIGluIHIuZmFjZXM6CiAgICAgICAgICAgICAgICByZXN0b3JlZD1zZWxmLmZhY2VfcmVzdG9yZXIucmVzdG9yZShmLmNyb3Asc2VsZi5jb25maWcuZmlkZWxpdHkpCiAgICAgICAgICAgICAgICBpZiBub3QgaXNpbnN0YW5jZShyZXN0b3JlZCxJbWFnZS5JbWFnZSk6IHJhaXNlIFR5cGVFcnJvcigiZmFjZSByZXN0b3JlciBtdXN0IHJldHVybiBQSUwuSW1hZ2UiKQogICAgICAgICAgICAgICAgZi5jcm9wPV9saW1pdChyZXN0b3JlZC5jb252ZXJ0KCJSR0IiKSxzZWxmLmNvbmZpZy5tYXhfc2lkZSkKICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1siZmFjZV9yZXN0b3JhdGlvbiJdPShwZXJmX2NvdW50ZXIoKS10KSoxMDAwCiAgICAgICAgICAgIHQ9cGVyZl9jb3VudGVyKCk7IHIuZW5oYW5jZWQ9X2xpbWl0KGVuaGFuY2VkLHNlbGYuY29uZmlnLm1heF9zaWRlKTsgci5vdXRwdXRfZGltZW5zaW9ucz1bci5lbmhhbmNlZC53aWR0aCxyLmVuaGFuY2VkLmhlaWdodF0KICAgICAgICAgICAgci5zdGFnZV90aW1lc19tc1sib3V0cHV0X3NpemVfbGltaXQiXT0ocGVyZl9jb3VudGVyKCktdCkqMTAwMAogICAgICAgICAgICBpZiBub3Qgci5mYWNlcyBhbmQgc2VsZi5jb25maWcubm9fZmFjZV9wb2xpY3kgPT0gImVycm9yIjogcmFpc2UgUnVudGltZUVycm9yKCJub19mYWNlIikKICAgICAgICAgICAgci5zdGF0dXM9Im9rIiBpZiByLmZhY2VzIGVsc2UgIm5vX2ZhY2UiCiAgICAgICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOiByLnN0YXR1cyxyLmVycm9yPSJlcnJvciIsZiJ7dHlwZShlKS5fX25hbWVfX306IHtlfSIKICAgICAgICByZXR1cm4gcgoKZGVmIHByb2Nlc3NfYmF0Y2gocGlwZWxpbmU6IEltYWdlUGlwZWxpbmUsIHNvdXJjZXMpOgogICAgIiIiUHJvY2VzcyBldmVyeSBpbnB1dCBpbmRlcGVuZGVudGx5OyBhIGJhZCBmaWxlIGRvZXMgbm90IHN0b3AgdGhlIGJhdGNoLiIiIgogICAgcmV0dXJuIFsoc291cmNlLCBwaXBlbGluZS5wcm9jZXNzKHNvdXJjZSkpIGZvciBzb3VyY2UgaW4gc291cmNlc10K'))
source_setup = LOCAL_BUNDLE / 'environment' / 'setup_emoca_source.sh'
run_quiet(['bash', str(source_setup), str(LOCAL_BUNDLE), str(WORKDIR)],
          label='Pinned EMOCA source setup')
print('Bundle verified; project and pinned EMOCA source are ready.')


In [ ]:
# Build the isolated legacy EMOCA environment. Success logs are suppressed.
if not ACCEPT_FLAME_EMOCA_LICENSES:
    raise RuntimeError(
        'Asset download is disabled. Enable it only after personally accepting FLAME and EMOCA terms.'
    )

miniconda = WORKDIR / 'miniconda'
miniconda_conda = miniconda / 'bin' / 'conda'
if miniconda_conda.is_file():
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
elif shutil.which('conda') is None:
    installer = WORKDIR / 'miniconda.sh'
    run_quiet(['wget', '-q', 'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh', '-O', str(installer)],
              label='Miniconda download')
    run_quiet(['bash', str(installer), '-b', '-p', str(miniconda)],
              label='Miniconda installation')
    os.environ['PATH'] = str(miniconda / 'bin') + os.pathsep + os.environ.get('PATH', '')
    os.environ['CONDA_EXE'] = str(miniconda_conda)
else:
    os.environ.setdefault('CONDA_EXE', shutil.which('conda'))

setup_env = os.environ.copy()
setup_env['CONDA_EXE'] = setup_env.get('CONDA_EXE') or shutil.which('conda') or ''
if not setup_env['CONDA_EXE']:
    raise RuntimeError('Conda executable not found after Miniconda setup.')
setup_env['PATH'] = str(Path(setup_env['CONDA_EXE']).parent) + os.pathsep + setup_env.get('PATH', '')
setup_env.pop('PYTHONPATH', None)
env_setup = LOCAL_BUNDLE / 'environment' / 'setup_emoca_env.sh'
run_quiet(['bash', str(env_setup), str(LOCAL_BUNDLE), str(WORKDIR)],
          env=setup_env, label='EMOCA Python 3.8 / Torch 1.11 environment setup')

# Use the current minimal downloader embedded in this notebook. This avoids
# accidentally running the older, full downloader from a previously uploaded bundle.
import base64
asset_env = setup_env.copy()
asset_env['ACCEPT_FLAME_EMOCA_LICENSES'] = '1'
asset_setup = WORKDIR / 'download_emoca_assets_minimal.sh'
asset_setup.write_bytes(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgYmFzaApzZXQgLWV1byBwaXBlZmFpbAoKV09SS0RJUj0iJHsxOj91c2FnZTogZG93bmxvYWRfZW1vY2FfYXNzZXRzLnNoIFdPUktESVJ9IgppZiBbWyAiJHtBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM6LTF9IiAhPSAiMSIgXV07IHRoZW4KICBjYXQgPiYyIDw8J0VPRicKQmxvY2tlZCBieSBBQ0NFUFRfRkxBTUVfRU1PQ0FfTElDRU5TRVM9MC4gU2V0IGl0IHRvIDEgb25seSBhZnRlciBwZXJzb25hbGx5IGFjY2VwdGluZyB0aGUgRkxBTUUgYW5kIEVNT0NBIHRlcm1zLgpFT0YKICBleGl0IDEwCmZpCgpFTU9DQV9ST09UPSIkV09SS0RJUi9wcm9qZWN0L21vZGVsL2Vtb2NhIgpBU1NFVF9ST09UPSIkRU1PQ0FfUk9PVC9hc3NldHMiCnRlc3QgLWQgIiRFTU9DQV9ST09UL2dkbF9hcHBzL0VNT0NBL2RlbW9zIiB8fCB7IGVjaG8gIlBpbm5lZCBFTU9DQSBzb3VyY2UgaXMgbWlzc2luZy4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgd2dldCA+L2Rldi9udWxsIHx8IHsgZWNobyAid2dldCBpcyByZXF1aXJlZC4iID4mMjsgZXhpdCAyOyB9CmNvbW1hbmQgLXYgdW56aXAgPi9kZXYvbnVsbCB8fCB7IGVjaG8gInVuemlwIGlzIHJlcXVpcmVkLiIgPiYyOyBleGl0IDI7IH0KCiMgVGhlIGJlbmNobWFyayBvbmx5IGxvYWRzIEVNT0NBX3YyX2xyX21zZV8yMC4gUmVtb3ZlIHVudXNlZCBkZW1vIGRhdGEsCiMgbGVnYWN5L2FsdGVybmF0ZSBjaGVja3BvaW50cyBhbmQgc3RhbGUgYXJjaGl2ZXMgbGVmdCBieSB0aGUgZmFpbGVkIHJ1bi4Kcm0gLWYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQS56aXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX21wLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfY29zXzEuNS56aXAiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YS56aXAiIFwKICAiJEFTU0VUX1JPT1QvREVDQS56aXAiICIkQVNTRVRfUk9PVC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIgpybSAtcmYgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9FTU9DQSIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbXAiIFwKICAiJEFTU0VUX1JPT1QvRU1PQ0EvbW9kZWxzL0VNT0NBX3YyX2xyX2Nvc18xLjUiIFwKICAiJEFTU0VUX1JPT1QvZGF0YS9FTU9DQV90ZXN0X2V4YW1wbGVfZGF0YSIKcm1kaXIgIiRBU1NFVF9ST09UL2RhdGEiIDI+L2Rldi9udWxsIHx8IHRydWUKCiMgUmVjbGFpbSBwYWNrYWdlIGNhY2hlcyBiZWZvcmUgZXh0cmFjdGluZyBhc3NldHMgaW50byBLYWdnbGUncyBsaW1pdGVkIGRpc2suCmlmIFtbIC1uICIke0NPTkRBX0VYRTotfSIgJiYgLXggIiR7Q09OREFfRVhFfSIgXV07IHRoZW4KICAiJENPTkRBX0VYRSIgY2xlYW4gLS1hbGwgLS15ZXMgPi9kZXYvbnVsbCAyPiYxIHx8IHRydWUKICAiJENPTkRBX0VYRSIgcnVuIC1uIHdvcmszOCBweXRob24gLW0gcGlwIGNhY2hlIHB1cmdlID4vZGV2L251bGwgMj4mMSB8fCB0cnVlCmZpCgpkb3dubG9hZF9leHRyYWN0KCkgewogIGxvY2FsIHVybD0iJDEiIGFyY2hpdmU9IiQyIiBkZXN0aW5hdGlvbj0iJDMiCiAgbWtkaXIgLXAgIiRkZXN0aW5hdGlvbiIgIiQoZGlybmFtZSAiJGFyY2hpdmUiKSIKICBybSAtZiAiJGFyY2hpdmUiICIkYXJjaGl2ZS5wYXJ0IgogIHdnZXQgLXEgIiR1cmwiIC1PICIkYXJjaGl2ZS5wYXJ0IgogIG12ICIkYXJjaGl2ZS5wYXJ0IiAiJGFyY2hpdmUiCiAgdW56aXAgLXEgLW8gIiRhcmNoaXZlIiAtZCAiJGRlc3RpbmF0aW9uIgogIHJtIC1mICIkYXJjaGl2ZSIKfQoKQkFTRV9VUkw9Imh0dHBzOi8vZG93bmxvYWQuaXMudHVlLm1wZy5kZS9lbW9jYS9hc3NldHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwLnppcCIgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscyIKZG93bmxvYWRfZXh0cmFjdCAiJEJBU0VfVVJML0VNT0NBL21vZGVscy9ERUNBLnppcCIgXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvREVDQS56aXAiICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9ERUNBLnppcCIgIiRBU1NFVF9ST09UL0RFQ0EuemlwIiAiJEFTU0VUX1JPT1QiCmRvd25sb2FkX2V4dHJhY3QgIiRCQVNFX1VSTC9GYWNlUmVjb2duaXRpb24uemlwIiAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uLnppcCIgIiRBU1NFVF9ST09UIgpkb3dubG9hZF9leHRyYWN0ICIkQkFTRV9VUkwvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QvRkxBTUUuemlwIiAiJEFTU0VUX1JPT1QiCgpmb3IgcmVxdWlyZWQgaW4gXAogICIkQVNTRVRfUk9PVC9FTU9DQS9tb2RlbHMvRU1PQ0FfdjJfbHJfbXNlXzIwIiBcCiAgIiRBU1NFVF9ST09UL0VNT0NBL21vZGVscy9ERUNBIiBcCiAgIiRBU1NFVF9ST09UL0RFQ0EiIFwKICAiJEFTU0VUX1JPT1QvRmFjZVJlY29nbml0aW9uIiBcCiAgIiRBU1NFVF9ST09UL0ZMQU1FIjsgZG8KICB0ZXN0IC1kICIkcmVxdWlyZWQiIHx8IHsgZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXQgd2FzIG5vdCBleHRyYWN0ZWQ6ICRyZXF1aXJlZCIgPiYyOyBleGl0IDM7IH0KZG9uZQoKZWNobyAiUmVxdWlyZWQgRU1PQ0EgYXNzZXRzIHJlYWR5OyB1bnVzZWQgY2hlY2twb2ludHMgYW5kIGRlbW8gZGF0YSBza2lwcGVkLiIK'))
run_quiet(['bash', str(asset_setup), str(WORKDIR)], env=asset_env,
          label='FLAME/EMOCA asset download')
print('EMOCA environment and required assets are ready.')


In [ ]:
# Two-image smoke pass. The output contains only the metric summary.
SMOKE = DRIVE_RESULTS / 'smoke'
run_quiet([sys.executable, str(SCRIPTS / 'launch_workers.py'),
           '--bundle-root', str(LOCAL_BUNDLE), '--workdir', str(WORKDIR),
           '--output-dir', str(SMOKE), '--limit', '2', '--gpus', str(GPU_COUNT)],
          label='Two-image smoke inference')
subprocess.run([sys.executable, str(SCRIPTS / 'summarize.py'),
                '--bundle-root', str(LOCAL_BUNDLE), '--results-dir', str(SMOKE),
                '--sample-name', 'smoke-2-images', '--expected-images', '2'], check=True)


In [ ]:
# Full test (3,068 images); JSONL rows and summaries are written directly to Drive.
FULL = DRIVE_RESULTS / 'full'
if RUN_FULL_BENCHMARK:
    run_quiet([sys.executable, str(SCRIPTS / 'launch_workers.py'),
               '--bundle-root', str(LOCAL_BUNDLE), '--workdir', str(WORKDIR),
               '--output-dir', str(FULL), '--gpus', str(GPU_COUNT)],
              label='Full RAF-DB test inference')
    subprocess.run([sys.executable, str(SCRIPTS / 'summarize.py'),
                    '--bundle-root', str(LOCAL_BUNDLE), '--results-dir', str(FULL),
                    '--sample-name', 'full-test-3068'], check=True)
    print(f'Results saved in: {FULL}')
else:
    print(f'Full run skipped. Existing results: {FULL}')
